In [ ]:
# ============================================================
# CELL 1 — IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import io
import base64

In [ ]:
# ============================================================
# CELL 2 — LOAD DATASET
# ============================================================
df_raw = pd.read_csv('Bengaluru_House_Data.csv')
print(f'Dataset loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')
print('\nColumn names:', df_raw.columns.tolist())
print('\nFirst 5 rows:')
display(df_raw.head())
print('\nMissing values per column:')
display(df_raw.isnull().sum())

In [ ]:
# ============================================================
# CELL 3 — DATA CLEANING
# ============================================================
df = df_raw.copy()

# --- 3a. Drop columns not useful for prediction ---
df.drop(columns=['society', 'availability'], inplace=True)

# --- 3b. Drop rows with missing critical values ---
df.dropna(subset=['location', 'size', 'bath', 'price'], inplace=True)

# --- 3c. Fill missing balcony with median ---
df['balcony'].fillna(df['balcony'].median(), inplace=True)

# --- 3d. Parse 'size' (e.g. '2 BHK', '3 Bedroom') → integer BHK ---
def parse_bhk(s):
    try:
        return int(str(s).split()[0])
    except:
        return np.nan

df['bhk'] = df['size'].apply(parse_bhk)
df.dropna(subset=['bhk'], inplace=True)
df['bhk'] = df['bhk'].astype(int)

# --- 3e. Parse 'total_sqft' — handle ranges like '2100-2850' ---
def parse_sqft(s):
    s = str(s).strip()
    if '-' in s:
        parts = s.split('-')
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except:
            return np.nan
    try:
        return float(s)
    except:
        return np.nan

df['total_sqft'] = df['total_sqft'].apply(parse_sqft)
df.dropna(subset=['total_sqft'], inplace=True)

# --- 3f. Derived feature: price per sqft ---
df['price_per_sqft'] = df['price'] * 1e5 / df['total_sqft']

# --- 3g. Strip and normalise location names ---
df['location'] = df['location'].str.strip().str.lower()

# Group rare locations (< 10 listings) as 'other'
loc_counts = df['location'].value_counts()
rare_locs = loc_counts[loc_counts < 10].index
df['location'] = df['location'].apply(lambda x: 'other' if x in rare_locs else x)

# --- 3h. Remove outliers using price_per_sqft within each location ---
def remove_pps_outliers(df_in):
    df_out = pd.DataFrame()
    for loc, sub in df_in.groupby('location'):
        m  = np.mean(sub['price_per_sqft'])
        sd = np.std(sub['price_per_sqft'])
        reduced = sub[(sub['price_per_sqft'] >= m - sd) & (sub['price_per_sqft'] <= m + sd)]
        df_out = pd.concat([df_out, reduced], ignore_index=True)
    return df_out

df = remove_pps_outliers(df)

# --- 3i. Remove cases where #bathrooms > bhk+2 (data anomalies) ---
df = df[df['bath'] < df['bhk'] + 2]

# --- 3j. Remove extreme price outliers (1st–99th percentile) ---
p1, p99 = df['price'].quantile([0.01, 0.99])
df = df[(df['price'] >= p1) & (df['price'] <= p99)]

# --- 3k. Normalise area_type ---
df['area_type'] = df['area_type'].str.strip()

# Final dtypes
df['bath']    = df['bath'].astype(int)
df['balcony'] = df['balcony'].astype(int)

df.reset_index(drop=True, inplace=True)
print(f'Clean dataset: {df.shape[0]} rows × {df.shape[1]} columns')
display(df.head())
print('\nData types:')
display(df.dtypes)

In [ ]:
# ============================================================
# CELL 4 — FEATURE ENGINEERING & ENCODING
# ============================================================
# One-hot encode location (already grouped rare ones as 'other')
dummies = pd.get_dummies(df['location'], drop_first=True)

# Label-encode area_type
le_area = LabelEncoder()
df['area_type_enc'] = le_area.fit_transform(df['area_type'])

# Build feature matrix
feature_cols_base = ['total_sqft', 'bath', 'balcony', 'bhk', 'area_type_enc']
X = pd.concat([df[feature_cols_base], dummies], axis=1)
y = df['price']    # in lakhs

print(f'Feature matrix shape: {X.shape}')
print(f'Target (price) stats:')
display(y.describe())

# Save column order and encoders for the prediction function
FEATURE_COLUMNS = X.columns.tolist()
LOCATION_LIST   = sorted(df['location'].unique().tolist())
AREA_TYPE_LIST  = sorted(df['area_type'].unique().tolist())

In [ ]:
# ============================================================
# CELL 5 — MODEL TRAINING & EVALUATION  (BACKEND)
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

models = {
    'Linear Regression'       : LinearRegression(),
    'Ridge Regression'        : Ridge(alpha=10),
    'Lasso Regression'        : Lasso(alpha=1),
    'Random Forest'           : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting'       : GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    if name in ('Linear Regression', 'Ridge Regression', 'Lasso Regression'):
        model.fit(X_train_sc, y_train)
        preds = model.predict(X_test_sc)
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
    mae  = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)
    results[name] = {'MAE': round(mae, 2), 'RMSE': round(rmse, 2), 'R²': round(r2, 4)}
    print(f'{name:30s} | MAE={mae:.2f}  RMSE={rmse:.2f}  R²={r2:.4f}')

results_df = pd.DataFrame(results).T
print('\n=== Model Comparison ===')
display(results_df)

# Select best model by R²
best_name  = results_df['R²'].astype(float).idxmax()
best_model = models[best_name]
print(f'\n✅  Best model: {best_name}  (R² = {results_df.loc[best_name, "R²"]})')

In [ ]:
# ============================================================
# CELL 6 — SAVE ARTEFACTS
# ============================================================
joblib.dump(best_model,        'house_price_model.pkl')
joblib.dump(scaler,            'scaler.pkl')
joblib.dump(le_area,           'label_encoder_area.pkl')
joblib.dump(FEATURE_COLUMNS,   'feature_columns.pkl')
joblib.dump(LOCATION_LIST,     'location_list.pkl')
joblib.dump(AREA_TYPE_LIST,    'area_type_list.pkl')
print('Saved: house_price_model.pkl, scaler.pkl, label_encoder_area.pkl')
print('Saved: feature_columns.pkl, location_list.pkl, area_type_list.pkl')

In [ ]:
# ============================================================
# CELL 7 — PREDICTION HELPER  (BACKEND FUNCTION)
# ============================================================
def predict_price(location, area_type, total_sqft, bath, balcony, bhk):
    """
    Returns predicted price in Lakhs (INR).
    Parameters
    ----------
    location   : str   — neighbourhood name (must be in LOCATION_LIST or 'other')
    area_type  : str   — one of AREA_TYPE_LIST
    total_sqft : float — total square feet
    bath       : int   — number of bathrooms
    balcony    : int   — number of balconies
    bhk        : int   — number of bedrooms
    """
    loc = location.strip().lower()
    if loc not in LOCATION_LIST:
        loc = 'other'

    area_enc = le_area.transform([area_type])[0]

    # Build base feature row
    row = pd.Series(0.0, index=FEATURE_COLUMNS)
    row['total_sqft']    = float(total_sqft)
    row['bath']          = int(bath)
    row['balcony']       = int(balcony)
    row['bhk']           = int(bhk)
    row['area_type_enc'] = int(area_enc)

    # Set location dummy if it appears as a column
    if loc in FEATURE_COLUMNS:
        row[loc] = 1.0

    X_input = pd.DataFrame([row])

    if best_name in ('Linear Regression', 'Ridge Regression', 'Lasso Regression'):
        X_sc = scaler.transform(X_input)
        price = best_model.predict(X_sc)[0]
    else:
        price = best_model.predict(X_input)[0]

    return round(max(price, 0), 2)

In [ ]:
# ============================================================
# CELL 8 — EDA CHARTS  (helper)
# ============================================================
def fig_to_img_widget(fig):
    """Convert a matplotlib figure to an ipywidgets Image widget."""
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=100)
    plt.close(fig)
    buf.seek(0)
    return widgets.Image(value=buf.read(), format='png',
                         layout=widgets.Layout(width='100%', max_width='720px'))

def eda_charts():
    charts = []

    # 1. Price distribution
    fig, ax = plt.subplots(figsize=(7, 3.5))
    sns.histplot(df['price'], bins=50, kde=True, color='steelblue', ax=ax)
    ax.set_title('Price Distribution (Lakhs)', fontsize=13, fontweight='bold')
    ax.set_xlabel('Price (Lakhs)')
    charts.append(fig_to_img_widget(fig))

    # 2. BHK vs Price boxplot
    fig, ax = plt.subplots(figsize=(7, 3.5))
    bhk_order = sorted(df['bhk'].unique())
    sns.boxplot(data=df, x='bhk', y='price', order=bhk_order,
                palette='Blues', ax=ax)
    ax.set_title('Price by BHK', fontsize=13, fontweight='bold')
    ax.set_xlabel('BHK'); ax.set_ylabel('Price (Lakhs)')
    charts.append(fig_to_img_widget(fig))

    # 3. Correlation heatmap
    fig, ax = plt.subplots(figsize=(6, 4))
    corr_cols = ['total_sqft', 'bath', 'balcony', 'bhk', 'price']
    sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f',
                cmap='coolwarm', ax=ax)
    ax.set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
    charts.append(fig_to_img_widget(fig))

    # 4. Model R² comparison
    fig, ax = plt.subplots(figsize=(7, 3.5))
    names = list(results.keys())
    r2s   = [results[n]['R²'] for n in names]
    colors = ['#2ecc71' if n == best_name else '#3498db' for n in names]
    bars = ax.barh(names, r2s, color=colors)
    ax.bar_label(bars, fmt='%.4f', padding=3)
    ax.set_xlim(0, 1.05)
    ax.set_title('Model R² Comparison', fontsize=13, fontweight='bold')
    ax.set_xlabel('R²')
    charts.append(fig_to_img_widget(fig))

    # 5. Scatter: actual vs predicted
    if best_name in ('Linear Regression', 'Ridge Regression', 'Lasso Regression'):
        y_pred_all = best_model.predict(scaler.transform(X_test))
    else:
        y_pred_all = best_model.predict(X_test)

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.scatter(y_test, y_pred_all, alpha=0.4, color='steelblue', s=15)
    mn, mx = y_test.min(), y_test.max()
    ax.plot([mn, mx], [mn, mx], 'r--', linewidth=1.5)
    ax.set_title('Actual vs Predicted', fontsize=13, fontweight='bold')
    ax.set_xlabel('Actual Price (Lakhs)')
    ax.set_ylabel('Predicted Price (Lakhs)')
    charts.append(fig_to_img_widget(fig))

    return charts

In [ ]:
# ============================================================
# CELL 9 — FRONTEND  (ipywidgets)
# ============================================================

# ------------ style helpers ---------------------------------
CARD_STYLE = """
<style>
  .pred-card {
    background: linear-gradient(135deg, #1a73e8, #0d47a1);
    color: white;
    border-radius: 12px;
    padding: 24px 32px;
    font-family: 'Segoe UI', sans-serif;
    max-width: 480px;
    margin: 16px 0;
  }
  .pred-card h2 { margin: 0 0 4px 0; font-size: 14px; opacity: 0.85; }
  .pred-card .amount { font-size: 36px; font-weight: 700; margin: 8px 0; }
  .pred-card .sub { font-size: 12px; opacity: 0.75; }
  .pred-card .info-row { margin-top: 12px; font-size: 13px; opacity: 0.9; }
  .error-card {
    background: #fdecea; border-left: 4px solid #d32f2f;
    border-radius: 6px; padding: 12px 16px;
    color: #b71c1c; font-family: 'Segoe UI', sans-serif;
    max-width: 480px; margin: 16px 0;
  }
  .app-title {
    font-family: 'Segoe UI', sans-serif;
    background: #0d47a1;
    color: white;
    padding: 16px 24px;
    border-radius: 10px;
    margin-bottom: 8px;
  }
  .app-title h1 { margin: 0; font-size: 22px; }
  .app-title p  { margin: 4px 0 0 0; font-size: 13px; opacity: 0.85; }
</style>
"""

display(HTML(CARD_STYLE))

# ------------ title banner ----------------------------------
display(HTML("""
<div class='app-title'>
  <h1>🏠 Bengaluru House Price Predictor</h1>
  <p>Fill in the property details below to get an estimated market price.</p>
</div>
"""))

# ------------ input widgets ---------------------------------
W = widgets.Layout(width='320px')

w_location = widgets.Combobox(
    placeholder='e.g. whitefield',
    options=LOCATION_LIST,
    description='Location:',
    ensure_option=False,
    style={'description_width': '110px'},
    layout=W
)
w_area_type = widgets.Dropdown(
    options=AREA_TYPE_LIST,
    description='Area Type:',
    style={'description_width': '110px'},
    layout=W
)
w_sqft = widgets.FloatText(
    value=1200.0,
    description='Total Sq.Ft:',
    step=50,
    style={'description_width': '110px'},
    layout=W
)
w_bhk = widgets.IntSlider(
    value=2, min=1, max=10,
    description='BHK:',
    style={'description_width': '110px'},
    layout=W
)
w_bath = widgets.IntSlider(
    value=2, min=1, max=10,
    description='Bathrooms:',
    style={'description_width': '110px'},
    layout=W
)
w_balcony = widgets.IntSlider(
    value=1, min=0, max=5,
    description='Balconies:',
    style={'description_width': '110px'},
    layout=W
)

btn_predict = widgets.Button(
    description='  Predict Price',
    button_style='primary',
    icon='home',
    layout=widgets.Layout(width='180px', height='40px', margin='12px 0 0 0')
)
btn_eda = widgets.Button(
    description='  Show EDA Charts',
    button_style='info',
    icon='bar-chart',
    layout=widgets.Layout(width='200px', height='40px', margin='12px 0 0 8px')
)
btn_clear = widgets.Button(
    description='  Clear',
    button_style='warning',
    icon='refresh',
    layout=widgets.Layout(width='120px', height='40px', margin='12px 0 0 8px')
)

out_result = widgets.Output()
out_eda    = widgets.Output()

# ------------ two-column form layout -----------------------
col1 = widgets.VBox([w_location, w_area_type, w_sqft])
col2 = widgets.VBox([w_bhk, w_bath, w_balcony])
form = widgets.HBox([col1, widgets.Box(layout=widgets.Layout(width='32px')), col2])
buttons = widgets.HBox([btn_predict, btn_eda, btn_clear])

# ------------ callbacks ------------------------------------
def on_predict(b):
    with out_result:
        clear_output(wait=True)
        try:
            loc  = w_location.value.strip().lower() or 'other'
            at   = w_area_type.value
            sqft = float(w_sqft.value)
            bath = int(w_bath.value)
            bal  = int(w_balcony.value)
            bhk  = int(w_bhk.value)

            if sqft <= 0:
                display(HTML("<div class='error-card'>⚠️ Total Sq.Ft must be > 0.</div>"))
                return

            price = predict_price(loc, at, sqft, bath, bal, bhk)
            crore = price / 100

            display(HTML(f"""
            <div class='pred-card'>
              <h2>ESTIMATED PRICE</h2>
              <div class='amount'>₹ {price:,.2f} L</div>
              <div class='sub'>(≈ ₹ {crore:.4f} Cr &nbsp;|&nbsp; Model: {best_name})</div>
              <div class='info-row'>
                📍 {loc.title()} &nbsp;·&nbsp; {bhk} BHK &nbsp;·&nbsp;
                {sqft:,.0f} sq.ft &nbsp;·&nbsp;
                {bath} Bath &nbsp;·&nbsp; {bal} Balcony
              </div>
            </div>
            """))
        except Exception as e:
            display(HTML(f"<div class='error-card'>❌ Error: {e}</div>"))

def on_eda(b):
    with out_eda:
        clear_output(wait=True)
        display(HTML("<b style='font-family:Segoe UI;font-size:16px;'>📊 Exploratory Data Analysis</b>"))
        charts = eda_charts()
        for c in charts:
            display(c)

def on_clear(b):
    with out_result:
        clear_output()
    with out_eda:
        clear_output()

btn_predict.on_click(on_predict)
btn_eda.on_click(on_eda)
btn_clear.on_click(on_clear)

# ------------ render UI ------------------------------------
display(form, buttons, out_result, out_eda)